# 2. Store Selection

Cross-tabulate `store.csv` on StoreType / Promo2 / CompetitionDistance and select a stratified set of 6 stores that covers meaningful variation on those axes, restricted to stores with (nearly) complete sales history. Corresponds to step 2 of the workflow in `CLAUDE.md`; this is the scoping decision that lets the paper analyze forecast performance by store characteristic instead of using the full 1,115-store dataset.

In [ ]:
import sys, json
from pathlib import Path

SRC = Path.cwd().parent / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from data import load_store, load_train
from store_selection import cross_tab, select_stores, filter_candidates
from plotting import set_paper_style, sequential_cmap, comma_axis, CATEGORICAL, INK_PRIMARY, INK_MUTED, SURFACE

set_paper_style()

OUTPUTS = Path.cwd().parent / "outputs"
FIGURES = OUTPUTS / "figures"
(OUTPUTS / "tables").mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

store = load_store()
train = load_train()

## Cross-tabulation: StoreType x Promo2 x CompetitionDistance tier

In [ ]:
cross_tab(store)

### Visualizing the cross-tab

Darker cells are more common combinations; lighter/blank cells are rarer - this is why a hand-picked, stratified set of stores is needed instead of a random sample, which would likely miss the rare combinations entirely.

In [ ]:
ct = cross_tab(store).rename(columns=lambda c: "unknown" if pd.isna(c) else c)
ct.index = [f"{st} / Promo2={p2}" for st, p2 in ct.index]

fig, ax = plt.subplots(figsize=(6, 4.5))
sns.heatmap(
    ct,
    annot=True,
    fmt="d",
    cmap=sequential_cmap(),
    linewidths=2,
    linecolor=SURFACE,
    cbar_kws={"label": "number of stores"},
    ax=ax,
)
# text contrast: dark ink on light cells, white on dark cells
threshold = ct.values.max() * 0.55
for text, value in zip(ax.texts, ct.values.flatten()):
    text.set_color("white" if value > threshold else INK_PRIMARY)
ax.set_xlabel("CompetitionDistance tier")
ax.set_ylabel("")
ax.set_title("Store counts by StoreType, Promo2, and distance tier")
fig.tight_layout()
fig.savefig(FIGURES / "store_crosstab_heatmap.png")
plt.show()

## Selecting 6 stores

`select_stores` first drops stores whose train.csv row count is below 95% of the full calendar span (incomplete history), then greedily picks stores to cover StoreType, Promo2, and CompetitionDistance-tier facets one at a time - each pick is the store that covers the most still-missing facet, with ties broken toward a brand-new (StoreType, Promo2, tier) combination. This guarantees every StoreType appears at least once (the primary axis) while still spending the remaining picks on Promo2/distance contrast.

In [ ]:
result = select_stores(store, train, n=6)
result.justification

In [ ]:
result.justification.to_csv(OUTPUTS / "tables" / "store_selection.csv", index=False)
with open(OUTPUTS / "selected_stores.json", "w") as f:
    json.dump(result.store_ids, f)
result.store_ids

### Where the 6 picks sit in the full population

All stores that passed the completeness / known-distance filter, plotted by CompetitionDistance and grouped by StoreType; the 6 selected stores are highlighted.

In [ ]:
rng = np.random.default_rng(42)
pool = filter_candidates(store, train)

store_types = sorted(pool["StoreType"].unique())
y_pos = {st: i for i, st in enumerate(store_types)}
pool = pool.assign(
    y=pool["StoreType"].astype(str).map(y_pos).astype(float) + rng.uniform(-0.18, 0.18, size=len(pool)),
    Selected=pool["Store"].isin(result.store_ids),
)

bg = pool.loc[~pool["Selected"]]
fg = pool.loc[pool["Selected"]]

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(bg["CompetitionDistance"], bg["y"], s=14, color=INK_MUTED, alpha=0.35, linewidths=0, label="Other stores")
ax.scatter(fg["CompetitionDistance"], fg["y"], s=70, color=CATEGORICAL[0], edgecolor=SURFACE, linewidths=1.5, label="Selected (6)", zorder=3)
for _, row in fg.iterrows():
    ax.annotate(f"  {int(row['Store'])}", (row["CompetitionDistance"], row["y"]), fontsize=8, va="center")

ax.set_yticks(range(len(store_types)))
ax.set_yticklabels(store_types)
ax.set_xscale("log")
comma_axis(ax, "x")
ax.set_xlabel("CompetitionDistance (meters, log scale)")
ax.set_ylabel("StoreType")
ax.set_title("Selected stores against the full candidate population")
ax.legend(frameon=False, loc="upper left")
fig.tight_layout()
fig.savefig(FIGURES / "store_selection_scatter.png")
plt.show()

## Justification (for the paper)

Each selected store fills a gap in StoreType, Promo2, or CompetitionDistance-tier coverage that the previously-selected stores left open (see the `Reason` column above) - the first four picks establish one store per StoreType, and the last two add Promo2/distance contrast that repeating a StoreType allows. All six passed the completeness filter, so none require special handling of missing weeks beyond the routine dropped closed days.